Step 0 - Explanation

In [1]:
"""
Transformer Model: Masked Nucleotide Prediction
------------------------------------------------
Dataset: Salmonella Typhimurium (6 genomes)

Input files:
- train.txt   : 5 genomes, raw sequences (targets)
- train_m.txt : 5 genomes, masked sequences (inputs)
- test.txt    : Salmonella 6, raw sequences (targets)
- test_m.txt  : Salmonella 6, masked sequences (inputs)

Masking Strategy:
- 5-mer rotation masking at 20% (1 in every 5 bases)
- Masked positions replaced with 'M'

Model Architecture:
- Custom Transformer Encoder
- Embedding dim : 128
- Attention heads: 4
- Encoder layers: 2
- Window size   : 512
- Vocabulary    : {A, C, G, T, M, PAD}

Training:
- Optimiser : Adam
- Loss      : Cross Entropy (masked positions only)
- Device    : CUDA if available, else CPU
"""

"\nTransformer Model: Masked Nucleotide Prediction\n------------------------------------------------\nDataset: Salmonella Typhimurium (6 genomes)\n\nInput files:\n- train.txt   : 5 genomes, raw sequences (targets)\n- train_m.txt : 5 genomes, masked sequences (inputs)\n- test.txt    : Salmonella 6, raw sequences (targets)\n- test_m.txt  : Salmonella 6, masked sequences (inputs)\n\nMasking Strategy:\n- 5-mer rotation masking at 20% (1 in every 5 bases)\n- Masked positions replaced with 'M'\n\nModel Architecture:\n- Custom Transformer Encoder\n- Embedding dim : 128\n- Attention heads: 4\n- Encoder layers: 2\n- Window size   : 512\n- Vocabulary    : {A, C, G, T, M, PAD}\n\nTraining:\n- Optimiser : Adam\n- Loss      : Cross Entropy (masked positions only)\n- Device    : CUDA if available, else CPU\n"

Step 1 - Imports

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import itertools
from collections import Counter


Step 2 - Hyperparameters and Configuration

In [3]:
bases   = ['A', 'C', 'G', 'T']
kmers   = [''.join(k) for k in itertools.product(bases, repeat=5)]  # 1024 5-mers
kmer2idx = {kmer: idx for idx, kmer in enumerate(kmers)}             # ATGCA -> 0 etc

# Special tokens
kmer2idx['MASK'] = 1024   # masked 5-mer token
kmer2idx['PAD']  = 1025   # padding token
kmer2idx['UNK']  = 1026   # unknown 5-mer (contains N)

VOCAB      = kmer2idx
VOCAB_SIZE = len(VOCAB)   # 1027

# ── Windowing ─────────────────────────────────────────────────────────────────
WINDOW_SIZE   = 128    # now means 128 5-mers = 640 bases of context
STRIDE        = 64

# ── Model Architecture ────────────────────────────────────────────────────────
EMBED_DIM     = 128
NUM_HEADS     = 4
NUM_LAYERS    = 3
FF_DIM        = 256
DROPOUT       = 0.1

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 64
EPOCHS        = 20
LEARNING_RATE = 3e-4

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Vocabulary size : {VOCAB_SIZE}")
print(f"  Using device    : {DEVICE}")

  Vocabulary size : 1027
  Using device    : cuda


Step 3 - Dataset and Windowing

In [ ]:
class DNADataset(Dataset):
    def __init__(self, original_path, window_size=WINDOW_SIZE, stride=STRIDE):
        self.window_size = window_size
        self.stride      = stride
        self.windows_input  = []
        self.windows_target = []

        original_seqs = open(original_path, "r").readlines()

        for original_seq in original_seqs:
            original_seq = original_seq.strip()

            # ── Convert sequence to 5-mer tokens ─────────────────────────
            kmers = []
            for i in range(0, len(original_seq) - 4, 1):  # sliding window of 5
                kmer = original_seq[i:i+5]
                if 'N' in kmer:
                    kmers.append(VOCAB['UNK'])
                else:
                    kmers.append(VOCAB.get(kmer, VOCAB['UNK']))

            # ── Mask every 5th kmer (rotation masking) ────────────────────
            masked_kmers = kmers.copy()
            for i in range(len(masked_kmers)):
                if i % 5 == (i // 5) % 5:  # rotation pattern
                    masked_kmers[i] = VOCAB['MASK']

            # ── Slice into windows ────────────────────────────────────────
            for start in range(0, len(kmers) - window_size, stride):
                input_window  = masked_kmers[start : start + window_size]
                target_window = kmers       [start : start + window_size]

                self.windows_input.append(torch.tensor(input_window,  dtype=torch.long))
                self.windows_target.append(torch.tensor(target_window, dtype=torch.long))

    def __len__(self):
        return len(self.windows_input)

    def __getitem__(self, idx):
        return self.windows_input[idx], self.windows_target[idx]


# ── Load datasets ─────────────────────────────────────────────────────────────
train_dataset = DNADataset("train.txt")
test_dataset  = DNADataset("test.txt")

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"  Training windows : {len(train_dataset):,}")
print(f"  Test windows     : {len(test_dataset):,}")

In [5]:
with open("train.txt") as f:
    seq = f.read().replace("\n", "")

counts = Counter(seq)
total = len(seq)
for base, count in sorted(counts.items()):
    print(f"  {base}: {count:,} ({count/total*100:.1f}%)")

  A: 5,883,392 (23.9%)
  C: 6,401,505 (26.0%)
  G: 6,406,549 (26.1%)
  T: 5,886,833 (24.0%)


Step 4 - Transformer Model

In [6]:
class DNATransformer(nn.Module):
    def __init__(self):
        super(DNATransformer, self).__init__()

        # Embedding layer: converts 5-mer token IDs to vectors
        # 1027 possible tokens (1024 5-mers + MASK + PAD + UNK)
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=VOCAB['PAD'])

        # Positional encoding: tells the model where each 5-mer is in the window
        self.pos_embedding = nn.Embedding(WINDOW_SIZE, EMBED_DIM)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = EMBED_DIM,
            nhead           = NUM_HEADS,
            dim_feedforward = FF_DIM,
            dropout         = DROPOUT,
            batch_first     = True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS)

        # Output layer: predicts one of VOCAB_SIZE 5-mer tokens at each position
        self.output_layer = nn.Linear(EMBED_DIM, VOCAB_SIZE)

    def forward(self, x):
        # x shape: (batch, window_size)

        # Create position indices [0, 1, 2, ..., window_size-1]
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)

        # 5-mer token embeddings + positional embeddings
        x = self.embedding(x) + self.pos_embedding(positions)

        # Pass through transformer encoder
        x = self.transformer(x)

        # Project to vocabulary size (1027)
        x = self.output_layer(x)

        return x  # shape: (batch, window_size, vocab_size)


# ── Initialise model ──────────────────────────────────────────────────────────
model = DNATransformer().to(DEVICE)
print(model)
print(f"\n  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

DNATransformer(
  (embedding): Embedding(1027, 128, padding_idx=1025)
  (pos_embedding): Embedding(128, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_layer): Linear(in_features=128, out_features=1027, bias=True)
)

  Total parameters: 677,763


Step 5 - Training Loop

In [7]:
# Loss & Optimiser
criterion = nn.CrossEntropyLoss(ignore_index=VOCAB['PAD'])
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# ── Learning rate scheduler ───────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

def train(model, loader, optimizer, criterion):
    model.train()
    total_loss    = 0
    total_correct = 0
    total_masked  = 0

    for input_kmers, target_kmers in loader:
        input_kmers  = input_kmers.to(DEVICE)
        target_kmers = target_kmers.to(DEVICE)

        optimizer.zero_grad()

        output = model(input_kmers)  # (batch, window_size, vocab_size)

        # ── Only compute loss on masked positions ─────────────────────────
        mask           = (input_kmers == VOCAB['MASK'])
        masked_output  = output[mask]         # (num_masked, vocab_size)
        masked_targets = target_kmers[mask]   # (num_masked,)

        loss = criterion(masked_output, masked_targets)

        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        predictions    = masked_output.argmax(dim=-1)
        total_correct += (predictions == masked_targets).sum().item()
        total_masked  += mask.sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / total_masked * 100 if total_masked > 0 else 0
    return avg_loss, accuracy


# ── Run Training ──────────────────────────────────────────────────────────────
best_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    print(f"  Epoch {epoch:02d}/{EPOCHS} | Loss: {train_loss:.4f} | Masked Accuracy: {train_acc:.2f}%")

    scheduler.step(train_loss)

    if train_loss < best_loss:
        best_loss = train_loss
        torch.save(model.state_dict(), "transformer_best_5mer.pth")
        print(f"  --> Best model saved at epoch {epoch}")

  Epoch 01/20 | Loss: 4.2705 | Masked Accuracy: 30.09%
  --> Best model saved at epoch 1
  Epoch 02/20 | Loss: 0.1971 | Masked Accuracy: 94.40%
  --> Best model saved at epoch 2
  Epoch 03/20 | Loss: 0.0579 | Masked Accuracy: 98.13%
  --> Best model saved at epoch 3
  Epoch 04/20 | Loss: 0.0434 | Masked Accuracy: 98.45%
  --> Best model saved at epoch 4
  Epoch 05/20 | Loss: 0.0393 | Masked Accuracy: 98.53%
  --> Best model saved at epoch 5
  Epoch 06/20 | Loss: 0.0375 | Masked Accuracy: 98.56%
  --> Best model saved at epoch 6
  Epoch 07/20 | Loss: 0.0364 | Masked Accuracy: 98.58%
  --> Best model saved at epoch 7
  Epoch 08/20 | Loss: 0.0356 | Masked Accuracy: 98.59%
  --> Best model saved at epoch 8
  Epoch 09/20 | Loss: 0.0350 | Masked Accuracy: 98.60%
  --> Best model saved at epoch 9
  Epoch 10/20 | Loss: 0.0345 | Masked Accuracy: 98.61%
  --> Best model saved at epoch 10
  Epoch 11/20 | Loss: 0.0340 | Masked Accuracy: 98.62%
  --> Best model saved at epoch 11
  Epoch 12/20 | Los

Step 6 - Evaluation

In [10]:
# ── Full Evaluation on Test Set (Salmonella 6 - unseen genome) ────────────────
model.eval()

total_correct = 0
total_masked  = 0
total_loss    = 0

# Per position accuracy tracking
position_correct = torch.zeros(WINDOW_SIZE).to(DEVICE)
position_total   = torch.zeros(WINDOW_SIZE).to(DEVICE)

with torch.no_grad():
    for input_kmers, target_kmers in test_loader:
        input_kmers  = input_kmers.to(DEVICE)
        target_kmers = target_kmers.to(DEVICE)

        output = model(input_kmers)  # (batch, window_size, vocab_size)

        # ── Loss on masked positions ──────────────────────────────────────
        mask           = (input_kmers == VOCAB['MASK'])
        masked_output  = output[mask]
        masked_targets = target_kmers[mask]

        loss           = criterion(masked_output, masked_targets)
        total_loss    += loss.item()

        # ── Overall accuracy ──────────────────────────────────────────────
        predictions    = output.argmax(dim=-1)
        total_correct += (predictions[mask] == target_kmers[mask]).sum().item()
        total_masked  += mask.sum().item()

        # ── Per position accuracy ─────────────────────────────────────────
        for pos in range(WINDOW_SIZE):
            pos_mask = mask[:, pos]
            if pos_mask.sum() > 0:
                position_correct[pos] += (predictions[:, pos][pos_mask] == target_kmers[:, pos][pos_mask]).sum()
                position_total[pos]   += pos_mask.sum()

# ── Results ───────────────────────────────────────────────────────────────────
test_accuracy = total_correct / total_masked * 100
test_loss     = total_loss / len(test_loader)
train_accuracy = 98.71  # from training
generalisation_gap = train_accuracy - test_accuracy

print("=" * 55)
print("  EVALUATION RESULTS — Salmonella 6 (Unseen Genome)")
print("=" * 55)
print(f"  Test Loss              : {test_loss:.4f}")
print(f"  Test Accuracy          : {test_accuracy:.2f}%")
print(f"  Train Accuracy         : {train_accuracy:.2f}%")
print(f"  Generalisation Gap     : {generalisation_gap:.2f}%")
print(f"  Total masked positions : {total_masked:,}")
print("=" * 55)

# ── Per rotation position accuracy ────────────────────────────────────────────
print("\n  Accuracy per rotation position (1-5):")
rotation_accs = []
for rot in range(5):
    positions = list(range(rot, WINDOW_SIZE, 5))
    correct   = position_correct[positions].sum().item()
    total     = position_total[positions].sum().item()
    acc       = correct / total * 100 if total > 0 else 0
    rotation_accs.append(acc)
    print(f"  Rotation {rot+1}: {acc:.2f}%")

print(f"\n  Mean rotation accuracy : {sum(rotation_accs)/len(rotation_accs):.2f}%")

  EVALUATION RESULTS — Salmonella 6 (Unseen Genome)
  Test Loss              : 0.0317
  Test Accuracy          : 98.65%
  Train Accuracy         : 98.71%
  Generalisation Gap     : 0.06%
  Total masked positions : 1,942,938

  Accuracy per rotation position (1-5):
  Rotation 1: 97.19%
  Rotation 2: 98.93%
  Rotation 3: 97.21%
  Rotation 4: 100.00%
  Rotation 5: 100.00%

  Mean rotation accuracy : 98.67%
